<a href="https://colab.research.google.com/github/srinjaysaha1605/YTShortsMaker/blob/main/YTShortsMaker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INSTALL DEPENDENCIES**

---

In [ ]:
# @title
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================

print("🚀 Setting up YTShortsMaker V2...\n")

!pip -q install google-genai edge-tts faster-whisper

print("\n✅ Python packages installed.")

# FFmpeg
import subprocess

result = subprocess.run(
    ["ffmpeg", "-version"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

if result.returncode == 0:
    print("✅ FFmpeg available.")
else:
    print("⚠️ FFmpeg not found.")

# **MOUNT DRIVE**

---



In [ ]:
# @title
# ============================================================
# CELL 2 — GOOGLE DRIVE + PROJECT SYSTEM
# ============================================================

from google.colab import drive
import os
import json
from datetime import datetime

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------

print("☁️ Connecting to Google Drive...")

drive.mount("/content/drive")

# ------------------------------------------------------------
# Main YTShortsMaker directory
# ------------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/YTShortsMaker_V2"

os.makedirs(BASE_DIR, exist_ok=True)

print(f"📁 Base directory:")
print(BASE_DIR)

# ------------------------------------------------------------
# Create / Load Project
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("📂 PROJECT")
print("=" * 60)

print("""
[1] 🆕 Create new project
[2] 📂 Open existing project
""")

project_choice = input("Choose (1/2): ").strip()

# ------------------------------------------------------------
# NEW PROJECT
# ------------------------------------------------------------

if project_choice == "1":

    PROJECT_NAME = input(
        "\n🎯 Enter project name: "
    ).strip()

    if not PROJECT_NAME:
        raise ValueError("❌ Project name cannot be empty.")

    # Remove characters that can cause path problems
    PROJECT_NAME = "".join(
        c for c in PROJECT_NAME
        if c not in '<>:"/\\|?*'
    ).strip()

    PROJECT_DIR = os.path.join(
        BASE_DIR,
        PROJECT_NAME
    )

    # Prevent accidental overwrite
    if os.path.exists(PROJECT_DIR):

        timestamp = datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )

        PROJECT_DIR = os.path.join(
            BASE_DIR,
            f"{PROJECT_NAME}_{timestamp}"
        )

    # Create project structure
    INPUT_DIR = os.path.join(
        PROJECT_DIR,
        "input"
    )

    IMAGE_DIR = os.path.join(
        PROJECT_DIR,
        "images"
    )

    OUTPUT_DIR = os.path.join(
        PROJECT_DIR,
        "output"
    )

    SCENE_DIR = os.path.join(
        OUTPUT_DIR,
        "scenes"
    )

    for directory in [
        PROJECT_DIR,
        INPUT_DIR,
        IMAGE_DIR,
        OUTPUT_DIR,
        SCENE_DIR
    ]:
        os.makedirs(directory, exist_ok=True)

    # Project metadata
    project_data = {
        "project_name": PROJECT_NAME,
        "created": datetime.now().isoformat(),
        "status": "created"
    }

    PROJECT_JSON = os.path.join(
        PROJECT_DIR,
        "project.json"
    )

    with open(
        PROJECT_JSON,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            project_data,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("\n✅ New project created!")
    print(f"📁 {PROJECT_DIR}")

# ------------------------------------------------------------
# EXISTING PROJECT
# ------------------------------------------------------------

elif project_choice == "2":

    projects = [
        name for name in os.listdir(BASE_DIR)
        if os.path.isdir(
            os.path.join(BASE_DIR, name)
        )
    ]

    if not projects:
        raise FileNotFoundError(
            "❌ No existing projects found."
        )

    print("\n📂 Existing projects:\n")

    for i, project in enumerate(
        projects,
        start=1
    ):
        print(f"[{i}] {project}")

    project_index = int(
        input("\nSelect project: ")
    ) - 1

    if (
        project_index < 0
        or project_index >= len(projects)
    ):
        raise ValueError(
            "❌ Invalid project selection."
        )

    PROJECT_NAME = projects[
        project_index
    ]

    PROJECT_DIR = os.path.join(
        BASE_DIR,
        PROJECT_NAME
    )

    INPUT_DIR = os.path.join(
        PROJECT_DIR,
        "input"
    )

    IMAGE_DIR = os.path.join(
        PROJECT_DIR,
        "images"
    )

    OUTPUT_DIR = os.path.join(
        PROJECT_DIR,
        "output"
    )

    SCENE_DIR = os.path.join(
        OUTPUT_DIR,
        "scenes"
    )

    # Ensure folders exist
    for directory in [
        INPUT_DIR,
        IMAGE_DIR,
        OUTPUT_DIR,
        SCENE_DIR
    ]:
        os.makedirs(directory, exist_ok=True)

    print("\n✅ Existing project loaded!")
    print(f"📁 {PROJECT_DIR}")

else:

    raise ValueError(
        "❌ Please choose either 1 or 2."
    )

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("📂 PROJECT READY")
print("=" * 60)

print(f"""
Project : {PROJECT_NAME}

📁 Project
   ├── input/
   ├── images/
   └── output/
       └── scenes/

Location:
{PROJECT_DIR}
""")

# **GEMINI API**

---



In [ ]:
# @title
# ============================================================
# CELL 3 — GEMINI API CONFIGURATION
# ============================================================

import os
from getpass import getpass
from google import genai

print("=" * 60)
print("🧠 GEMINI CONFIGURATION")
print("=" * 60)

# ------------------------------------------------------------
# API Key
# ------------------------------------------------------------

GEMINI_API_KEY = getpass(
    "\n🔑 Enter your Gemini API Key: "
)

if not GEMINI_API_KEY.strip():
    raise ValueError(
        "❌ Gemini API key cannot be empty."
    )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# ------------------------------------------------------------
# Initialize Gemini
# ------------------------------------------------------------

client = genai.Client(
    api_key=GEMINI_API_KEY
)

MODEL_NAME = "gemini-3.5-flash-lite"

print("\n✅ Gemini initialized.")
print(f"🤖 Model: {MODEL_NAME}")
print("🔐 API key stored only in the current Colab session.")

# **CHECK AVAILABLE MODELS**

---



In [ ]:
# @title
# ============================================================
# CHECK AVAILABLE GEMINI MODELS
# ============================================================

print("=" * 60)
print("🤖 AVAILABLE GEMINI MODELS")
print("=" * 60)

for model in client.models.list():
    if "generateContent" in model.supported_actions:
        print(f"✅ {model.name}")

# **SCRIPT INPUT**

---



In [ ]:
# @title
# ============================================================
# CELL 4 — SCRIPT INPUT / CUSTOM SCRIPT PROCESSOR
# ============================================================

import os
import json
import re

print("=" * 60)
print("✍️ SCRIPT INPUT")
print("=" * 60)

print("""
Choose your script source:

[1] 🤖 Generate script with Gemini
[2] 📝 Use my own script
""")

SCRIPT_MODE = input(
    "Choose (1-2): "
).strip()

if SCRIPT_MODE not in ["1", "2"]:
    raise ValueError("❌ Choose 1 or 2.")

# ------------------------------------------------------------
# CUSTOM SCRIPT INPUT
# ------------------------------------------------------------

if SCRIPT_MODE == "2":

    print("""
============================================================
📝 PASTE YOUR SCRIPT
============================================================

You can paste a messy script containing:

[HOOK]
[0:03-0:10]
Scene headings
Visual instructions
Text overlay instructions
CTA labels
etc.

Gemini will clean it and turn it into a
natural TTS-ready voiceover under 55 seconds.

When finished, press ENTER on an empty line.
============================================================
""")

    lines = []

    while True:

        line = input()

        if line.strip() == "":
            break

        lines.append(line)

    raw_script = "\n".join(lines).strip()

    if not raw_script:
        raise ValueError(
            "❌ No script was entered."
        )

    print("\n🧹 Cleaning and optimizing script...")

# ------------------------------------------------------------
# GEMINI SCRIPT GENERATION
# ------------------------------------------------------------

else:

    topic = input(
        "\n🎯 Enter your Short topic: "
    ).strip()

    if not topic:
        raise ValueError(
            "❌ Topic cannot be empty."
        )

    raw_script = f"""
Create a YouTube Short script about:

{topic}
"""

    print("\n🧠 Generating script with Gemini...")

# ------------------------------------------------------------
# SCRIPT PROCESSING PROMPT
# ------------------------------------------------------------

prompt = f"""
You are an expert viral YouTube Shorts scriptwriter.

Transform the following material into a concise,
natural-sounding voiceover for a YouTube Short.

SOURCE MATERIAL:
{raw_script}

STRICT REQUIREMENTS:

1. FINAL voiceover must be approximately 50–55 seconds.
2. Maximum 125 words.
3. NEVER exceed 130 words.
4. Keep the most interesting and important facts.
5. Start with a strong hook.
6. Conversational Grade-7 English.
7. Short, punchy sentences.
8. No greeting.
9. No "Welcome", "Hey guys", etc.
10. Do not invent facts.
11. Do not add information that isn't supported by
    the source material.
12. Keep important names, numbers and dates accurate.

IMPORTANT:

Return ONLY the spoken voiceover.

DO NOT include:

- TITLE:
- HOOK:
- VOICEOVER:
- SCRIPT:
- SCENE 1:
- SCENE 2:
- [timestamps]
- [HOOK]
- [CTA]
- Visual directions
- Camera directions
- Editing instructions
- Text overlay instructions
- Parentheses describing visuals
- Markdown
- Bullet points
- Quotation marks around the entire script

The result must be ready to send directly to a
text-to-speech engine.

SOURCE:
{raw_script}
"""

# ------------------------------------------------------------
# GENERATE CLEAN SCRIPT
# ------------------------------------------------------------

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)

clean_script = response.text.strip()

# ------------------------------------------------------------
# EXTRA SAFETY CLEANING
# ------------------------------------------------------------

# Remove markdown formatting
clean_script = re.sub(
    r"\*\*(.*?)\*\*",
    r"\1",
    clean_script
)

clean_script = re.sub(
    r"\*(.*?)\*",
    r"\1",
    clean_script
)

# Remove common labels if Gemini ignores instructions
clean_script = re.sub(
    r"^(TITLE|HOOK|VOICEOVER|SCRIPT|NARRATION)\s*:\s*",
    "",
    clean_script,
    flags=re.IGNORECASE
)

# Remove [HOOK], [CTA], timestamps etc.
clean_script = re.sub(
    r"\[[^\]]*\]",
    "",
    clean_script
)

# Remove common visual/editing instructions in parentheses
clean_script = re.sub(
    r"\([^)]*(?:text|overlay|visual|cut|camera|screen|show)[^)]*\)",
    "",
    clean_script,
    flags=re.IGNORECASE
)

# Clean excessive whitespace
clean_script = re.sub(
    r"\s+",
    " ",
    clean_script
).strip()

# ------------------------------------------------------------
# WORD COUNT
# ------------------------------------------------------------

word_count = len(
    clean_script.split()
)

print("\n" + "=" * 60)
print("🎬 FINAL TTS-READY SCRIPT")
print("=" * 60)

print(f"\n📝 Word count: {word_count}")

print("\n" + "-" * 60)
print(clean_script)
print("-" * 60)

# ------------------------------------------------------------
# SAFETY CHECK
# ------------------------------------------------------------

if word_count > 130:

    print(
        "\n⚠️ Script is still over 130 words."
    )

    print(
        "Running one final compression pass..."
    )

    compression_prompt = f"""
Shorten this YouTube Shorts voiceover to
NO MORE THAN 125 WORDS.

Keep the hook, key facts, important names,
numbers and dates.

Do not invent information.

Return ONLY the spoken narration.
No labels. No markdown. No explanations.

SCRIPT:
{clean_script}
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=compression_prompt
    )

    clean_script = response.text.strip()

    word_count = len(
        clean_script.split()
    )

    print(
        f"\n✅ Compressed to {word_count} words."
    )

# ------------------------------------------------------------
# SAVE SCRIPT
# ------------------------------------------------------------

SCRIPT_PATH = os.path.join(
    OUTPUT_DIR,
    "script.json"
)

script_data = {
    "source_mode": (
        "gemini"
        if SCRIPT_MODE == "1"
        else "custom"
    ),
    "script": clean_script,
    "word_count": word_count,
    "target_duration": "50-55 seconds",
    "max_words": 130
}

with open(
    SCRIPT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        script_data,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 60)
print("✅ SCRIPT READY")
print("=" * 60)

print(
    f"\n💾 Saved to:\n{SCRIPT_PATH}"
)

print(
    "\n➡️ Next: Cell 5"
)

# **SCENE PLANNING**

---



In [ ]:
# @title
# ============================================================
# CELL 5 — AI SCENE PLANNER
# ============================================================

import os
import json

print("=" * 60)
print("🎬 AI SCENE PLANNER")
print("=" * 60)

# ------------------------------------------------------------
# Load saved script
# ------------------------------------------------------------

if not os.path.exists(SCRIPT_PATH):
    raise FileNotFoundError(
        "❌ script.json not found. Run Cell 4 first."
    )

with open(
    SCRIPT_PATH,
    "r",
    encoding="utf-8"
) as f:
    script_data = json.load(f)

script_text = script_data["script"]

print("\n🧠 Analyzing script...")
print("🔍 Creating visual scene plan...")

# ------------------------------------------------------------
# Gemini prompt
# ------------------------------------------------------------

prompt = f"""
You are a professional YouTube Shorts video editor.

Analyze the following voiceover script and divide it into
logical visual scenes.

VOICEOVER:
{script_text}

RULES:

1. Create between 3 and 8 scenes depending on the script length.
2. Every word of the voiceover must belong to exactly one scene.
3. Keep sentences or natural phrases together where possible.
4. Do not rewrite or change the voiceover.
5. Each scene must have:
   - scene_number
   - voiceover
   - visual_description
6. The visual description must describe the EXACT type of image
   that would best represent that section.
7. Avoid generic descriptions.
8. Make the visuals interesting and suitable for YouTube Shorts.
9. Do not generate image prompts yet.
10. Return ONLY valid JSON.

JSON FORMAT:

{{
  "scenes": [
    {{
      "scene_number": 1,
      "voiceover": "...",
      "visual_description": "..."
    }}
  ]
}}
"""

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=prompt
)

# ------------------------------------------------------------
# Parse JSON
# ------------------------------------------------------------

raw_response = response.text.strip()

# Remove accidental markdown code fences
raw_response = raw_response.replace(
    "```json",
    ""
).replace(
    "```",
    ""
).strip()

try:

    scene_plan = json.loads(
        raw_response
    )

except json.JSONDecodeError:

    print("\n❌ Gemini returned invalid JSON.")
    print("\nRaw response:")
    print(raw_response)

    raise

# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

if "scenes" not in scene_plan:
    raise ValueError(
        "❌ Scene plan does not contain 'scenes'."
    )

scenes = scene_plan["scenes"]

if not scenes:
    raise ValueError(
        "❌ No scenes were generated."
    )

# ------------------------------------------------------------
# Save scene plan
# ------------------------------------------------------------

SCENE_PLAN_PATH = os.path.join(
    PROJECT_DIR,
    "scene_plan.json"
)

with open(
    SCENE_PLAN_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        scene_plan,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# Display scene plan
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("🎬 SCENE PLAN")
print("=" * 60)

for scene in scenes:

    print(
        f"\nSCENE {scene['scene_number']}"
    )

    print(
        f"🎙️ {scene['voiceover']}"
    )

    print(
        f"🖼️ {scene['visual_description']}"
    )

    print("-" * 60)

print(
    f"\n✅ {len(scenes)} scenes created."
)

print(
    f"💾 Saved to:\n{SCENE_PLAN_PATH}"
)

# **OPTIONAL CLEANUP LAYER**

---



In [ ]:
# @title
# ============================================================
# CELL 5A — PROJECT CLEANUP
# ============================================================

import os
import shutil

print("=" * 60)
print("🧹 PROJECT CLEANUP")
print("=" * 60)

# ------------------------------------------------------------
# Folders/files that are safe to regenerate
# ------------------------------------------------------------

CLEANUP_ITEMS = [
    os.path.join(PROJECT_DIR, "images"),
    os.path.join(PROJECT_DIR, "output"),
    os.path.join(PROJECT_DIR, "image_mapping.json"),
    os.path.join(PROJECT_DIR, "image_framing.json"),
    os.path.join(PROJECT_DIR, "scene_timing.json"),
    os.path.join(PROJECT_DIR, "scenes.json"),
    os.path.join(PROJECT_DIR, "scene_plan.json"),
    os.path.join(PROJECT_DIR, "captions.json"),
    os.path.join(PROJECT_DIR, "captions.srt"),
    os.path.join(PROJECT_DIR, "captions.ass"),
]

# ------------------------------------------------------------
# Show what will be removed
# ------------------------------------------------------------

existing_items = [
    item
    for item in CLEANUP_ITEMS
    if os.path.exists(item)
]

if not existing_items:

    print("\n✅ Project is already clean.")
    print("Nothing to remove.")

else:

    print(
        f"\n⚠️ Found {len(existing_items)} "
        "generated item(s):\n"
    )

    for item in existing_items:
        print(f"   • {os.path.basename(item)}")

    print("\nThese generated files/folders can be recreated.")

    confirm = input(
        "\nClean project? (y/n): "
    ).strip().lower()

    if confirm != "y":

        print("\n⏭️ Cleanup cancelled.")

    else:

        removed = 0

        for item in existing_items:

            try:

                if os.path.isdir(item):
                    shutil.rmtree(item)
                else:
                    os.remove(item)

                print(
                    f"🗑️ Removed: "
                    f"{os.path.basename(item)}"
                )

                removed += 1

            except Exception as e:

                print(
                    f"❌ Could not remove "
                    f"{os.path.basename(item)}: {e}"
                )

        print("\n" + "=" * 60)
        print("✅ CLEANUP COMPLETE")
        print("=" * 60)

        print(
            f"\n🗑️ Removed: {removed} item(s)"
        )

print("\n➡️ Next: Cell 6 — Upload Images")

# **IMAGE UPLOAD AND AUTOMATIC SCENE ORDERING**

---



In [ ]:
# @title
# ============================================================
# CELL 6 — IMAGE UPLOAD & AUTOMATIC IMAGE ORDERING
# ============================================================

from google.colab import files
from PIL import Image
import os
import re
import json
import shutil

print("=" * 60)
print("🖼️ IMAGE MANAGER")
print("=" * 60)

# ------------------------------------------------------------
# Load scene plan
# ------------------------------------------------------------

SCENE_PLAN_PATH = os.path.join(
    PROJECT_DIR,
    "scene_plan.json"
)

if not os.path.exists(SCENE_PLAN_PATH):
    raise FileNotFoundError(
        "❌ scene_plan.json not found. Run Cell 5 first."
    )

with open(
    SCENE_PLAN_PATH,
    "r",
    encoding="utf-8"
) as f:
    scene_plan = json.load(f)

scenes = scene_plan["scenes"]
script_scene_count = len(scenes)

print(
    f"\n🎬 Script scenes: {script_scene_count}"
)

print("""
ℹ️ IMAGE MODE

Images are independent of script scenes.

You can upload ANY number of images.

For example:
    5 script scenes
    7 images

The 7 images will simply become:

    Image 1
    Image 2
    Image 3
    Image 4
    Image 5
    Image 6
    Image 7

The later timing stage will distribute the
audio duration equally across these images.
""")

# ------------------------------------------------------------
# Show script requirements
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("📋 SCRIPT SCENES")
print("=" * 60)

for scene in scenes:

    print(f"""
Scene {scene['scene_number']}
🎙️ {scene['voiceover']}
🖼️ {scene['visual_description']}
""")

    print("-" * 60)

# ------------------------------------------------------------
# RESET IMAGE DIRECTORY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("🧹 IMAGE DIRECTORY")
print("=" * 60)

os.makedirs(
    IMAGE_DIR,
    exist_ok=True
)

IMAGE_EXTENSIONS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".bmp"
)

old_images = []

for filename in os.listdir(
    IMAGE_DIR
):

    path = os.path.join(
        IMAGE_DIR,
        filename
    )

    if (
        os.path.isfile(path)
        and filename.lower().endswith(
            IMAGE_EXTENSIONS
        )
    ):
        old_images.append(path)

if old_images:

    print(
        f"\n⚠️ Found {len(old_images)} "
        "previous image(s)."
    )

    print(
        "Clearing previous images..."
    )

    for path in old_images:
        os.remove(path)

    print(
        "✅ Old images cleared."
    )

else:

    print(
        "✅ No previous images found."
    )

# ------------------------------------------------------------
# Upload images
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("📤 UPLOAD IMAGES")
print("=" * 60)

print("""
Name your images numerically:

    1.jpg
    2.jpg
    3.jpg
    4.jpg
    5.jpg
    6.jpg
    7.jpg

You can also use descriptive names:

    1-monitor.jpg
    2-oled.png
    3-gaming.jpg

The number at the beginning determines
the visual order.

IMPORTANT:
The number of images DOES NOT need to match
the number of script scenes.
""")

uploaded = files.upload()

if not uploaded:

    raise RuntimeError(
        "❌ No images uploaded."
    )

# ------------------------------------------------------------
# Process uploaded images
# ------------------------------------------------------------

uploaded_images = []
used_numbers = set()

for filename in uploaded:

    source_path = filename

    # --------------------------------------------------------
    # Extract image order number
    # --------------------------------------------------------

    match = re.match(
        r"^\s*(\d+)",
        filename
    )

    if not match:

        print(
            f"\n⚠️ Skipped '{filename}'"
        )

        print(
            "   Filename must start with a number."
        )

        continue

    image_number = int(
        match.group(1)
    )

    # --------------------------------------------------------
    # Validate number
    # --------------------------------------------------------

    if image_number < 1:

        print(
            f"\n⚠️ Skipped '{filename}'"
        )

        print(
            "   Invalid image number."
        )

        continue

    # --------------------------------------------------------
    # Detect duplicate numbers
    # --------------------------------------------------------

    if image_number in used_numbers:

        print(
            f"\n❌ Skipped '{filename}'"
        )

        print(
            f"   Image number {image_number} "
            "is already being used."
        )

        continue

    # --------------------------------------------------------
    # Validate image
    # --------------------------------------------------------

    try:

        img = Image.open(
            source_path
        )

        img.load()

        width, height = img.size

    except Exception as e:

        print(
            f"\n❌ Skipped '{filename}'"
        )

        print(
            f"   Invalid image: {e}"
        )

        continue

    # --------------------------------------------------------
    # Create clean filename
    # --------------------------------------------------------

    extension = os.path.splitext(
        filename
    )[1].lower()

    clean_filename = (
        f"image_{image_number:03d}"
        f"{extension}"
    )

    destination = os.path.join(
        IMAGE_DIR,
        clean_filename
    )

    shutil.copy2(
        source_path,
        destination
    )

    used_numbers.add(
        image_number
    )

    uploaded_images.append({
        "number": image_number,
        "original": filename,
        "filename": clean_filename,
        "path": destination,
        "width": width,
        "height": height
    })

    print(
        f"\n✅ Image {image_number}: "
        f"{filename}"
    )

    print(
        f"   📐 {width}×{height}"
    )

    print(
        f"   💾 Saved as: "
        f"{clean_filename}"
    )

# ------------------------------------------------------------
# Sort images numerically
# ------------------------------------------------------------

uploaded_images.sort(
    key=lambda x: x["number"]
)

# ------------------------------------------------------------
# Build image mapping
# ------------------------------------------------------------

image_mapping = []

print("\n" + "=" * 60)
print("🎬 FINAL IMAGE ORDER")
print("=" * 60)

for index, image in enumerate(
    uploaded_images,
    start=1
):

    # IMPORTANT:
    # scene_number is retained for compatibility
    # with downstream cells.
    #
    # It now represents the VISUAL IMAGE SLOT,
    # NOT the number of script scenes.

    image_mapping.append({
        "scene_number": index,
        "image_index": index,
        "image_number": image["number"],
        "original": image["original"],
        "image": image["filename"],
        "path": image["path"],
        "width": image["width"],
        "height": image["height"]
    })

    print(
        f"\n🖼️ Image {index}"
    )

    print(
        f"   🔢 Original number: "
        f"{image['number']}"
    )

    print(
        f"   📁 {image['original']}"
    )

    print(
        f"   📐 {image['width']}×"
        f"{image['height']}"
    )

# ------------------------------------------------------------
# Save mapping
# ------------------------------------------------------------

IMAGE_MAPPING_PATH = os.path.join(
    PROJECT_DIR,
    "image_mapping.json"
)

mapping_data = {
    "script_scene_count": script_scene_count,
    "image_count": len(image_mapping),
    "images": image_mapping
}

with open(
    IMAGE_MAPPING_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        mapping_data,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("✅ IMAGE UPLOAD COMPLETE")
print("=" * 60)

print(
    f"\n🎬 Script scenes: "
    f"{script_scene_count}"
)

print(
    f"🖼️ Images uploaded: "
    f"{len(image_mapping)}"
)

print("""
📌 IMAGE / AUDIO LOGIC

The script and images are independent.

Example:

    5 script scenes
    7 images
    53 second audio

The next timing stage will calculate:

    53 ÷ 7 = ~7.57 seconds per image

Therefore:

    Image 1 → 0.00–7.57s
    Image 2 → 7.57–15.14s
    Image 3 → 15.14–22.71s
    Image 4 → 22.71–30.29s
    Image 5 → 30.29–37.86s
    Image 6 → 37.86–45.43s
    Image 7 → 45.43–53.00s
""")

print(
    f"\n💾 Mapping saved to:\n"
    f"{IMAGE_MAPPING_PATH}"
)

print(
    "\n➡️ Next: Cell 7 — 9:16 Image Preparation"
)

# **IMAGE PREPARATION**

---



In [ ]:
# @title
# ============================================================
# CELL 7 — 9:16 IMAGE PREPARATION
# ============================================================

from PIL import Image, ImageFilter, ImageOps
import os
import json
import numpy as np

print("=" * 60)
print("📐 IMAGE PREPARATION — 9:16")
print("=" * 60)

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

OUTPUT_WIDTH = 1080
OUTPUT_HEIGHT = 1920

print("""
Choose how your images should be placed inside the
9:16 YouTube Shorts canvas:

[1] 🚫  No Change — use original image
[2] ✂️  Smart Center Crop
[3] 🌫️  Fit + Blurred Background
[4] ⬛  Fit + Black Background
[5] 🖼️  Fit + White Background
[6] 📦  Original Image with Padding
""")

FRAME_MODE = input(
    "Choose (1-6): "
).strip()

if FRAME_MODE not in ["1", "2", "3", "4", "5", "6"]:
    raise ValueError(
        "❌ Invalid option. Choose 1-6."
    )

# ------------------------------------------------------------
# Load image mapping
# ------------------------------------------------------------

if not os.path.exists(IMAGE_MAPPING_PATH):
    raise FileNotFoundError(
        "❌ image_mapping.json not found. "
        "Run Cell 6 first."
    )

with open(
    IMAGE_MAPPING_PATH,
    "r",
    encoding="utf-8"
) as f:

    mapping_data = json.load(f)

scene_images = mapping_data["images"]

# ------------------------------------------------------------
# Create framed image directory
# ------------------------------------------------------------

FRAMED_DIR = os.path.join(
    IMAGE_DIR,
    "framed"
)

os.makedirs(
    FRAMED_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def fit_image(
    image,
    max_width,
    max_height
):

    image = image.copy()

    scale = min(
        max_width / image.width,
        max_height / image.height
    )

    new_size = (
        int(image.width * scale),
        int(image.height * scale)
    )

    return image.resize(
        new_size,
        Image.Resampling.LANCZOS
    )


def center_crop(
    image,
    target_ratio
):

    width, height = image.size
    current_ratio = width / height

    if current_ratio > target_ratio:

        # Too wide → crop left/right
        new_width = int(
            height * target_ratio
        )

        left = (
            width - new_width
        ) // 2

        return image.crop((
            left,
            0,
            left + new_width,
            height
        ))

    else:

        # Too tall → crop top/bottom
        new_height = int(
            width / target_ratio
        )

        top = (
            height - new_height
        ) // 2

        return image.crop((
            0,
            top,
            width,
            top + new_height
        ))


def blurred_background(
    image,
    width,
    height
):

    # Create background based on original image
    background = ImageOps.fit(
        image,
        (width, height),
        method=Image.Resampling.LANCZOS
    )

    # Strong blur
    background = background.filter(
        ImageFilter.GaussianBlur(35)
    )

    # Darken slightly
    background = background.convert(
        "RGBA"
    )

    overlay = Image.new(
        "RGBA",
        background.size,
        (0, 0, 0, 70)
    )

    background = Image.alpha_composite(
        background,
        overlay
    )

    # Fit original image inside
    foreground = fit_image(
        image,
        width,
        height
    ).convert("RGBA")

    x = (
        width - foreground.width
    ) // 2

    y = (
        height - foreground.height
    ) // 2

    background.alpha_composite(
        foreground,
        (x, y)
    )

    return background.convert("RGB")


def padded_image(
    image,
    width,
    height,
    background_color
):

    canvas = Image.new(
        "RGB",
        (width, height),
        background_color
    )

    foreground = fit_image(
        image,
        width,
        height
    )

    x = (
        width - foreground.width
    ) // 2

    y = (
        height - foreground.height
    ) // 2

    canvas.paste(
        foreground,
        (x, y)
    )

    return canvas


# ------------------------------------------------------------
# Process images
# ------------------------------------------------------------

processed_mapping = []

print("\n🎬 Processing scene images...\n")

for item in scene_images:

    scene_number = item["scene_number"]
    filename = item["image"]

    if filename is None:

        print(
            f"⚠️ Scene {scene_number}: "
            f"No image — skipped."
        )

        processed_mapping.append({
            "scene_number": scene_number,
            "original": None,
            "framed": None
        })

        continue

    source_path = os.path.join(
        IMAGE_DIR,
        filename
    )

    if not os.path.exists(source_path):

        print(
            f"❌ Scene {scene_number}: "
            f"{filename} not found."
        )

        processed_mapping.append({
            "scene_number": scene_number,
            "original": filename,
            "framed": None
        })

        continue

    # Open image
    image = Image.open(
        source_path
    ).convert("RGB")

    original_size = image.size

    # --------------------------------------------------------
    # Apply selected framing
    # --------------------------------------------------------

    if FRAME_MODE == "1":
      # No change
      image = Image.open(source_path).convert("RGB")

    elif FRAME_MODE == "2":
        # Center crop
        image = center_crop(
            image,
            OUTPUT_WIDTH / OUTPUT_HEIGHT
        )

        image = image.resize(
            (
                OUTPUT_WIDTH,
                OUTPUT_HEIGHT
            ),
            Image.Resampling.LANCZOS
        )

    elif FRAME_MODE == "3":

        # Fit + blurred background
        image = blurred_background(
            image,
            OUTPUT_WIDTH,
            OUTPUT_HEIGHT
        )

    elif FRAME_MODE == "4":

        # Black background
        image = padded_image(
            image,
            OUTPUT_WIDTH,
            OUTPUT_HEIGHT,
            (0, 0, 0)
        )

    elif FRAME_MODE == "5":

        # White background
        image = padded_image(
            image,
            OUTPUT_WIDTH,
            OUTPUT_HEIGHT,
            (255, 255, 255)
        )

    elif FRAME_MODE == "6":

        # Neutral dark background
        image = padded_image(
            image,
            OUTPUT_WIDTH,
            OUTPUT_HEIGHT,
            (20, 20, 20)
        )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    output_filename = (
        f"scene_{scene_number:02d}.jpg"
    )

    output_path = os.path.join(
        FRAMED_DIR,
        output_filename
    )

    image.save(
        output_path,
        "JPEG",
        quality=95
    )

    processed_mapping.append({
        "scene_number": scene_number,
        "original": filename,
        "framed": output_filename,
        "path": output_path,
        "original_size": original_size,
        "output_size": (
            OUTPUT_WIDTH,
            OUTPUT_HEIGHT
        )
    })

    print(
        f"✅ Scene {scene_number}: "
        f"{filename} → {output_filename}"
    )

# ------------------------------------------------------------
# Save framing information
# ------------------------------------------------------------

FRAMING_PATH = os.path.join(
    PROJECT_DIR,
    "image_framing.json"
)

framing_data = {
    "width": OUTPUT_WIDTH,
    "height": OUTPUT_HEIGHT,
    "mode": FRAME_MODE,
    "images": processed_mapping
}

with open(
    FRAMING_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        framing_data,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 60)
print("✅ IMAGE PREPARATION COMPLETE")
print("=" * 60)

print(
    f"\n📐 Output: "
    f"{OUTPUT_WIDTH} × {OUTPUT_HEIGHT}"
)

print(
    f"📁 Framed images:\n{FRAMED_DIR}"
)

print(
    f"\n💾 Settings saved:\n{FRAMING_PATH}"
)

# **PREVIEW PREPARED IMAGES**

---



In [ ]:
# @title
# ============================================================
# CELL 8 — PREVIEW ALL PREPARED IMAGES
# ============================================================

from PIL import Image
import matplotlib.pyplot as plt
import os
import json

print("=" * 60)
print("👀 PREVIEW — PREPARED IMAGES")
print("=" * 60)

# ------------------------------------------------------------
# Load latest framing data
# ------------------------------------------------------------

if not os.path.exists(FRAMING_PATH):
    raise FileNotFoundError(
        "❌ image_framing.json not found. "
        "Run Cell 7 first."
    )

with open(
    FRAMING_PATH,
    "r",
    encoding="utf-8"
) as f:
    framing_data = json.load(f)

images = framing_data.get("images", [])

print(
    f"\n🖼️ Images recorded in framing data: "
    f"{len(images)}"
)

# ------------------------------------------------------------
# Find all prepared images
# ------------------------------------------------------------

valid_images = []

for item in images:

    scene_number = item.get("scene_number")
    filename = item.get("framed")

    if not filename:
        print(
            f"⚠️ Image {scene_number}: "
            "No prepared image."
        )
        continue

    image_path = os.path.join(
        FRAMED_DIR,
        filename
    )

    if not os.path.exists(image_path):

        print(
            f"❌ Image {scene_number}: "
            f"{filename} not found."
        )

        continue

    valid_images.append({
        "number": scene_number,
        "filename": filename,
        "path": image_path
    })

# ------------------------------------------------------------
# Sort numerically
# ------------------------------------------------------------

valid_images.sort(
    key=lambda x: x["number"]
)

# ------------------------------------------------------------
# Display every image
# ------------------------------------------------------------

if not valid_images:

    raise RuntimeError(
        "❌ No prepared images available. "
        "Run Cell 7 first."
    )

print(
    f"\n✅ Found {len(valid_images)} "
    f"prepared image(s).\n"
)

for item in valid_images:

    image = Image.open(
        item["path"]
    )

    print("=" * 40)

    print(
        f"🖼️ IMAGE {item['number']}"
    )

    print(
        f"📁 {item['filename']}"
    )

    print(
        f"📐 {image.width} × {image.height}"
    )

    plt.figure(
        figsize=(5, 9)
    )

    plt.imshow(image)
    plt.axis("off")

    plt.title(
        f"Image {item['number']}"
    )

    plt.show()

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("✅ PREVIEW COMPLETE")
print("=" * 60)

print(
    f"\n🖼️ Total prepared images previewed: "
    f"{len(valid_images)}"
)

print("""
If everything looks good:
    → Continue to the automatic timing cell.

If something looks wrong:
    → Run Cell 7 again
    → Choose a different framing mode
    → Run this preview cell again
""")

# **AI VOICEOVER**

---



In [ ]:
# @title
# ============================================================
# CELL 9 — AI VOICEOVER
# ============================================================

import os
import asyncio
import edge_tts
from IPython.display import Audio, display

print("=" * 60)
print("🎙️ AI VOICEOVER")
print("=" * 60)

# ------------------------------------------------------------
# Load script
# ------------------------------------------------------------

if not os.path.exists(SCRIPT_PATH):
    raise FileNotFoundError(
        "❌ script.json not found. Run the script cell first."
    )

with open(
    SCRIPT_PATH,
    "r",
    encoding="utf-8"
) as f:
    script_data = json.load(f)

voiceover_text = script_data["script"]

print(f"\n📝 Script length: {len(voiceover_text.split())} words")

# ------------------------------------------------------------
# Voice options
# ------------------------------------------------------------

VOICES = {
    "1": {
        "name": "Andrew",
        "id": "en-US-AndrewMultilingualNeural"
    },
    "2": {
        "name": "Christopher",
        "id": "en-US-ChristopherNeural"
    },
    "3": {
        "name": "Eric",
        "id": "en-US-EricNeural"
    },
    "4": {
        "name": "Guy",
        "id": "en-US-GuyNeural"
    },
    "5": {
        "name": "Jenny",
        "id": "en-US-JennyNeural"
    },
    "6": {
        "name": "Aria",
        "id": "en-US-AriaNeural"
    }
}

print("\n🎤 Available voices:\n")

for key, voice in VOICES.items():
    print(
        f"[{key}] {voice['name']}"
    )

VOICE_CHOICE = input(
    "\nChoose voice (1-6): "
).strip()

if VOICE_CHOICE not in VOICES:
    raise ValueError(
        "❌ Invalid voice selection."
    )

selected_voice = VOICES[VOICE_CHOICE]["id"]

print(
    f"\n🎙️ Selected: "
    f"{VOICES[VOICE_CHOICE]['name']}"
)

# ------------------------------------------------------------
# Voice settings
# ------------------------------------------------------------

VOICE_RATE = "+2%"
VOICE_PITCH = "+0Hz"

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

VOICE_PATH = os.path.join(
    OUTPUT_DIR,
    "voiceover.mp3"
)

# ------------------------------------------------------------
# Generate voice
# ------------------------------------------------------------

async def generate_voice(
    text,
    output_file,
    voice
):

    communicate = edge_tts.Communicate(
        text=text,
        voice=voice,
        rate=VOICE_RATE,
        pitch=VOICE_PITCH
    )

    await communicate.save(
        output_file
    )

print("\n🎙️ Generating voiceover...")

await generate_voice(
    voiceover_text,
    VOICE_PATH,
    selected_voice
)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

if not os.path.exists(VOICE_PATH):

    raise RuntimeError(
        "❌ Voiceover file was not created."
    )

file_size = os.path.getsize(
    VOICE_PATH
)

print("\n" + "=" * 60)
print("✅ VOICEOVER GENERATED")
print("=" * 60)

print(
    f"\n🎤 Voice: "
    f"{VOICES[VOICE_CHOICE]['name']}"
)

print(
    f"📁 Saved to:\n{VOICE_PATH}"
)

print(
    f"💾 File size: "
    f"{file_size / 1024:.1f} KB"
)

# ------------------------------------------------------------
# Preview
# ------------------------------------------------------------

display(
    Audio(
        VOICE_PATH
    )
)

# **WHISPER TRANSCRIPTION + WORD TIMESTAMPS**

---



In [ ]:
# @title
# ============================================================
# CELL 10 — WHISPER TRANSCRIPTION + WORD TIMESTAMPS
# ============================================================

from faster_whisper import WhisperModel
import os
import json

print("=" * 60)
print("🧠 WHISPER TRANSCRIPTION")
print("=" * 60)

# ------------------------------------------------------------
# Check voiceover
# ------------------------------------------------------------

if not os.path.exists(VOICE_PATH):
    raise FileNotFoundError(
        "❌ voiceover.mp3 not found. Run Cell 9 first."
    )

# ------------------------------------------------------------
# Load Whisper
# ------------------------------------------------------------

print("\n🧠 Loading Whisper model...")

# CPU is intentional — avoids Colab CUDA compatibility issues.
# 'small' gives a good balance of speed and accuracy.
whisper_model = WhisperModel(
    "small",
    device="cpu",
    compute_type="int8"
)

print("✅ Whisper loaded on CPU.")

# ------------------------------------------------------------
# Transcribe
# ------------------------------------------------------------

print("\n🎧 Transcribing voiceover...")

segments, info = whisper_model.transcribe(
    VOICE_PATH,
    word_timestamps=True,
    vad_filter=True
)

# ------------------------------------------------------------
# Collect word timestamps
# ------------------------------------------------------------

words = []
segments_data = []

for segment in segments:

    segment_words = []

    if segment.words:

        for word in segment.words:

            clean_word = word.word.strip()

            if not clean_word:
                continue

            word_data = {
                "word": clean_word,
                "start": round(word.start, 3),
                "end": round(word.end, 3)
            }

            words.append(word_data)
            segment_words.append(word_data)

    segments_data.append({
        "start": round(segment.start, 3),
        "end": round(segment.end, 3),
        "text": segment.text.strip(),
        "words": segment_words
    })

# ------------------------------------------------------------
# Audio duration
# ------------------------------------------------------------

audio_duration = 0

if segments_data:
    audio_duration = max(
        segment["end"]
        for segment in segments_data
    )

# ------------------------------------------------------------
# Save timestamp data
# ------------------------------------------------------------

TIMESTAMPS_PATH = os.path.join(
    OUTPUT_DIR,
    "timestamps.json"
)

timestamp_data = {
    "language": info.language,
    "language_probability": round(
        info.language_probability,
        4
    ),
    "duration": round(
        audio_duration,
        3
    ),
    "word_count": len(words),
    "words": words,
    "segments": segments_data
}

with open(
    TIMESTAMPS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        timestamp_data,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("📝 TRANSCRIPTION")
print("=" * 60)

full_text = " ".join(
    word["word"]
    for word in words
)

print(f"\n{full_text}")

print("\n" + "-" * 60)
print("⏱️ WORD TIMESTAMPS")
print("-" * 60)

# Show first 30 words
for word in words[:30]:

    print(
        f"{word['start']:6.2f}s → "
        f"{word['end']:6.2f}s   "
        f"{word['word']}"
    )

if len(words) > 30:

    print(
        f"\n... and "
        f"{len(words) - 30} more words."
    )

print("\n" + "=" * 60)
print("✅ TRANSCRIPTION COMPLETE")
print("=" * 60)

print(
    f"\n🌐 Language: {info.language}"
)

print(
    f"⏱️ Duration: "
    f"{audio_duration:.2f}s"
)

print(
    f"📝 Words: "
    f"{len(words)}"
)

print(
    f"💾 Saved to:\n"
    f"{TIMESTAMPS_PATH}"
)

# **AUTOMATIC SHORT DURATION CONTROL**

---



In [ ]:
# @title
# ============================================================
# CELL 11 — AUTOMATIC SHORT DURATION CONTROL
# ============================================================

import os
import json
import asyncio
import edge_tts

print("=" * 60)
print("⏱️ SHORT DURATION CHECK")
print("=" * 60)

TARGET_MAX = 57.0
ABSOLUTE_MAX = 59.0

MAX_ATTEMPTS = 3

# ------------------------------------------------------------
# Load current script
# ------------------------------------------------------------

with open(
    SCRIPT_PATH,
    "r",
    encoding="utf-8"
) as f:
    script_data = json.load(f)

current_script = script_data["script"]

# ------------------------------------------------------------
# Helper — Generate TTS
# ------------------------------------------------------------

async def create_voiceover(text):

    communicate = edge_tts.Communicate(
        text=text,
        voice=selected_voice,
        rate=VOICE_RATE,
        pitch=VOICE_PITCH
    )

    await communicate.save(
        VOICE_PATH
    )

# ------------------------------------------------------------
# Helper — Get duration from Whisper timestamps
# ------------------------------------------------------------

def get_audio_duration():

    if not os.path.exists(TIMESTAMPS_PATH):
        return None

    with open(
        TIMESTAMPS_PATH,
        "r",
        encoding="utf-8"
    ) as f:
        data = json.load(f)

    return float(
        data.get("duration", 0)
    )

# ------------------------------------------------------------
# Compression loop
# ------------------------------------------------------------

for attempt in range(1, MAX_ATTEMPTS + 1):

    print(
        f"\n🔎 Attempt {attempt}/{MAX_ATTEMPTS}"
    )

    # --------------------------------------------------------
    # Generate voice
    # --------------------------------------------------------

    print("🎙️ Generating voiceover...")

    await create_voiceover(
        current_script
    )

    print("✅ Voice generated.")

    # --------------------------------------------------------
    # Run Whisper again
    # --------------------------------------------------------

    print("🧠 Measuring actual spoken duration...")

    segments, info = whisper_model.transcribe(
        VOICE_PATH,
        word_timestamps=True,
        vad_filter=True
    )

    words = []
    segments_data = []

    for segment in segments:

        segment_words = []

        if segment.words:

            for word in segment.words:

                clean_word = word.word.strip()

                if not clean_word:
                    continue

                word_data = {
                    "word": clean_word,
                    "start": round(word.start, 3),
                    "end": round(word.end, 3)
                }

                words.append(word_data)
                segment_words.append(word_data)

        segments_data.append({
            "start": round(segment.start, 3),
            "end": round(segment.end, 3),
            "text": segment.text.strip(),
            "words": segment_words
        })

    if not segments_data:
        raise RuntimeError(
            "❌ Whisper could not detect speech."
        )

    duration = max(
        segment["end"]
        for segment in segments_data
    )

    # --------------------------------------------------------
    # Save updated timestamps
    # --------------------------------------------------------

    timestamp_data = {
        "language": info.language,
        "language_probability": round(
            info.language_probability,
            4
        ),
        "duration": round(
            duration,
            3
        ),
        "word_count": len(words),
        "words": words,
        "segments": segments_data
    }

    with open(
        TIMESTAMPS_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            timestamp_data,
            f,
            indent=2,
            ensure_ascii=False
        )

    # --------------------------------------------------------
    # Check duration
    # --------------------------------------------------------

    print(
        f"\n⏱️ Actual duration: "
        f"{duration:.2f} seconds"
    )

    if duration <= TARGET_MAX:

        print(
            "\n✅ PERFECT — Short is within "
            f"{TARGET_MAX:.0f}-second target."
        )

        break

    # --------------------------------------------------------
    # Too long → compress
    # --------------------------------------------------------

    print(
        f"\n⚠️ Over {TARGET_MAX:.0f} seconds."
    )

    if duration >= ABSOLUTE_MAX:

        print(
            f"🚨 Audio is dangerously close to "
            f"{ABSOLUTE_MAX:.0f}+ seconds."
        )

    # Estimate required reduction
    reduction_ratio = TARGET_MAX / duration

    target_words = max(
        80,
        int(
            len(current_script.split())
            * reduction_ratio
            * 0.95
        )
    )

    print(
        f"✂️ Compressing to approximately "
        f"{target_words} words..."
    )

    compression_prompt = f"""
Rewrite this YouTube Shorts voiceover to fit
comfortably within approximately 50–55 seconds.

Current duration: {duration:.1f} seconds.
Target: under 55 seconds.
Maximum words: {target_words}.

Keep:
- The strongest hook
- The most important facts
- Important names
- Important numbers
- Important dates

Remove:
- Repetition
- Filler
- Less important details
- Unnecessary explanations

Do NOT invent facts.

Return ONLY the spoken narration.
No title.
No hook label.
No timestamps.
No visual instructions.
No markdown.

SCRIPT:
{current_script}
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=compression_prompt
    )

    current_script = response.text.strip()

    # --------------------------------------------------------
    # Update script JSON
    # --------------------------------------------------------

    script_data["script"] = current_script
    script_data["word_count"] = len(
        current_script.split()
    )

    with open(
        SCRIPT_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            script_data,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"✅ New script: "
        f"{script_data['word_count']} words"
    )

else:

    raise RuntimeError(
        "❌ Could not reduce the Short below "
        f"{TARGET_MAX} seconds after "
        f"{MAX_ATTEMPTS} attempts."
    )

# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("🎬 FINAL SHORT DURATION")
print("=" * 60)

print(
    f"\n⏱️ Duration: {duration:.2f}s"
)

print(
    f"📝 Words: "
    f"{len(current_script.split())}"
)

print(
    f"🎙️ Voice: "
    f"{VOICES[VOICE_CHOICE]['name']}"
)

print(
    f"\n💾 Script:\n{SCRIPT_PATH}"
)

print(
    f"\n💾 Voiceover:\n{VOICE_PATH}"
)

print(
    f"\n💾 Timestamps:\n{TIMESTAMPS_PATH}"
)

print("\n✅ Ready for caption generation.")

# **AUTOMATIC IMAGE → SCENE TIMING**

---



In [ ]:
# @title
# ============================================================
# CELL 12 — AUTOMATIC IMAGE → SCENE TIMING
# ============================================================

import os
import json
import re

print("=" * 60)
print("🎬 AUTOMATIC SCENE TIMING")
print("=" * 60)

# ------------------------------------------------------------
# Check required paths
# ------------------------------------------------------------

if not os.path.exists(OUTPUT_DIR):
    raise FileNotFoundError(
        f"❌ Output directory not found:\n{OUTPUT_DIR}"
    )

if not os.path.exists(TIMESTAMPS_PATH):
    raise FileNotFoundError(
        "❌ timestamps.json not found.\n"
        "Run Cell 10/11 first."
    )

# ------------------------------------------------------------
# Load actual audio duration
# ------------------------------------------------------------

with open(
    TIMESTAMPS_PATH,
    "r",
    encoding="utf-8"
) as f:

    timestamp_data = json.load(f)

audio_duration = float(
    timestamp_data["duration"]
)

# ------------------------------------------------------------
# Locate framed images
# ------------------------------------------------------------

# Cell 7 creates:
# IMAGE_DIR/
#     framed/
#
# Try the existing variable first.
# Otherwise reconstruct it.

if "FRAMED_DIR" in globals():

    framed_directory = FRAMED_DIR

else:

    framed_directory = os.path.join(
        IMAGE_DIR,
        "framed"
    )

if not os.path.exists(framed_directory):

    raise FileNotFoundError(
        f"❌ Framed image directory not found:\n"
        f"{framed_directory}\n\n"
        "Run Cell 7 first."
    )

# ------------------------------------------------------------
# Find prepared images
# ------------------------------------------------------------

files = []

for filename in os.listdir(
    framed_directory
):

    full_path = os.path.join(
        framed_directory,
        filename
    )

    if not os.path.isfile(full_path):
        continue

    # Accept common image formats
    if not filename.lower().endswith(
        (".jpg", ".jpeg", ".png", ".webp")
    ):
        continue

    files.append(filename)

# ------------------------------------------------------------
# Sort images naturally
# ------------------------------------------------------------

def natural_sort_key(filename):

    numbers = re.findall(
        r"\d+",
        filename
    )

    if numbers:
        return int(numbers[-1])

    return filename.lower()


files.sort(
    key=natural_sort_key
)

# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

if not files:

    raise RuntimeError(
        f"❌ No prepared images found in:\n"
        f"{framed_directory}\n\n"
        "Run Cell 7 again and make sure images are uploaded."
    )

image_count = len(files)

# ------------------------------------------------------------
# Calculate equal duration
# ------------------------------------------------------------

scene_duration = (
    audio_duration / image_count
)

print(
    f"\n🎙️ Audio duration: "
    f"{audio_duration:.2f}s"
)

print(
    f"🖼️ Images detected: "
    f"{image_count}"
)

print(
    f"⏱️ Duration per image: "
    f"{scene_duration:.2f}s"
)

# ------------------------------------------------------------
# Create scenes
# ------------------------------------------------------------

scenes = []

for index, filename in enumerate(
    files
):

    start_time = (
        index * scene_duration
    )

    if index == image_count - 1:

        end_time = audio_duration

    else:

        end_time = (
            (index + 1)
            * scene_duration
        )

    scene = {

        "scene_number": index + 1,

        "image": filename,

        "path": os.path.join(
            framed_directory,
            filename
        ),

        "start": round(
            start_time,
            3
        ),

        "end": round(
            end_time,
            3
        ),

        "duration": round(
            end_time - start_time,
            3
        )
    }

    scenes.append(scene)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

SCENES_PATH = os.path.join(
    OUTPUT_DIR,
    "scenes.json"
)

scenes_data = {

    "audio_duration": round(
        audio_duration,
        3
    ),

    "image_count": image_count,

    "scene_duration": round(
        scene_duration,
        3
    ),

    "timing_mode": "equal",

    "scenes": scenes
}

with open(
    SCENES_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        scenes_data,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("🎬 FINAL SCENE MAPPING")
print("=" * 60)

for scene in scenes:

    print(
        f"\nScene {scene['scene_number']}"
    )

    print(
        f"🖼️ {scene['image']}"
    )

    print(
        f"⏱️ "
        f"{scene['start']:.2f}s → "
        f"{scene['end']:.2f}s "
        f"({scene['duration']:.2f}s)"
    )

print("\n" + "=" * 60)
print("✅ SCENE TIMING COMPLETE")
print("=" * 60)

print(
    f"\n💾 Saved to:\n{SCENES_PATH}"
)

print(
    "\n➡️ Next: Caption generation"
)

# **SHORT-FORM CAPTION GENERATOR**

---



In [ ]:
# @title
# ============================================================
# CELL 13 — SHORT-FORM CAPTION GENERATOR
# ============================================================

import os
import json
import re

print("=" * 60)
print("📝 SHORT-FORM CAPTION GENERATION")
print("=" * 60)

# ------------------------------------------------------------
# Caption settings
# ------------------------------------------------------------

WORDS_PER_CAPTION = 4
MAX_CHARACTERS = 28
MIN_CAPTION_DURATION = 0.45
MAX_CAPTION_DURATION = 2.2

print(f"""
⚙️ Caption settings:

Words per caption: {WORDS_PER_CAPTION}
Maximum characters: {MAX_CHARACTERS}
Minimum duration: {MIN_CAPTION_DURATION}s
Maximum duration: {MAX_CAPTION_DURATION}s
""")

# ------------------------------------------------------------
# Load Whisper timestamps
# ------------------------------------------------------------

if not os.path.exists(TIMESTAMPS_PATH):

    raise FileNotFoundError(
        "❌ timestamps.json not found.\n"
        "Run Cell 10/11 first."
    )

with open(
    TIMESTAMPS_PATH,
    "r",
    encoding="utf-8"
) as f:

    timestamp_data = json.load(f)

words = timestamp_data.get(
    "words",
    []
)

if not words:

    raise RuntimeError(
        "❌ No word timestamps found."
    )

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def clean_word(word):

    return re.sub(
        r"\s+",
        " ",
        word.strip()
    )


def should_break(
    current_words,
    next_word
):

    if not current_words:
        return False

    # Maximum word count
    if len(current_words) >= WORDS_PER_CAPTION:
        return True

    current_text = " ".join(
        w["word"]
        for w in current_words
    )

    # Maximum character count
    test_text = (
        current_text
        + " "
        + next_word["word"]
    )

    if len(test_text) > MAX_CHARACTERS:
        return True

    # Natural punctuation break
    last_word = current_words[-1]["word"]

    if re.search(
        r"[.!?,:;]$",
        last_word
    ):
        return True

    return False


# ------------------------------------------------------------
# Build caption chunks
# ------------------------------------------------------------

captions = []
current = []

for word in words:

    word = {
        "word": clean_word(
            word["word"]
        ),
        "start": float(
            word["start"]
        ),
        "end": float(
            word["end"]
        )
    }

    if not word["word"]:
        continue

    if should_break(
        current,
        word
    ):

        captions.append(
            current
        )

        current = []

    current.append(
        word
    )

# Add final caption
if current:

    captions.append(
        current
    )

# ------------------------------------------------------------
# Convert chunks into caption events
# ------------------------------------------------------------

caption_events = []

for index, chunk in enumerate(
    captions,
    start=1
):

    start = chunk[0]["start"]
    end = chunk[-1]["end"]

    text = " ".join(
        word["word"]
        for word in chunk
    )

    duration = end - start

    # Avoid extremely short caption events
    if (
        duration < MIN_CAPTION_DURATION
        and caption_events
    ):

        previous = caption_events[-1]

        combined_text = (
            previous["text"]
            + " "
            + text
        )

        if len(combined_text) <= (
            MAX_CHARACTERS * 2
        ):

            previous["end"] = round(
                end,
                3
            )

            previous["duration"] = round(
                previous["end"]
                - previous["start"],
                3
            )

            previous["text"] = (
                combined_text
            )

            continue

    # Cap excessively long events
    if duration > MAX_CAPTION_DURATION:

        end = min(
            end,
            start + MAX_CAPTION_DURATION
        )

    caption_events.append({

        "index": len(caption_events) + 1,

        "start": round(
            start,
            3
        ),

        "end": round(
            end,
            3
        ),

        "duration": round(
            end - start,
            3
        ),

        "text": text
    })

# ------------------------------------------------------------
# Save JSON
# ------------------------------------------------------------

CAPTIONS_JSON_PATH = os.path.join(
    OUTPUT_DIR,
    "captions.json"
)

caption_data = {

    "count": len(
        caption_events
    ),

    "settings": {

        "words_per_caption":
            WORDS_PER_CAPTION,

        "max_characters":
            MAX_CHARACTERS,

        "min_duration":
            MIN_CAPTION_DURATION,

        "max_duration":
            MAX_CAPTION_DURATION
    },

    "captions":
        caption_events
}

with open(
    CAPTIONS_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        caption_data,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# Create SRT
# ------------------------------------------------------------

def seconds_to_srt(seconds):

    hours = int(
        seconds // 3600
    )

    minutes = int(
        (seconds % 3600) // 60
    )

    secs = int(
        seconds % 60
    )

    milliseconds = int(
        round(
            (seconds - int(seconds))
            * 1000
        )
    )

    if milliseconds >= 1000:

        milliseconds = 0
        secs += 1

    return (
        f"{hours:02d}:"
        f"{minutes:02d}:"
        f"{secs:02d},"
        f"{milliseconds:03d}"
    )


SRT_PATH = os.path.join(
    OUTPUT_DIR,
    "subtitles.srt"
)

with open(
    SRT_PATH,
    "w",
    encoding="utf-8"
) as f:

    for caption in caption_events:

        f.write(
            f"{caption['index']}\n"
        )

        f.write(
            f"{seconds_to_srt(caption['start'])}"
            f" --> "
            f"{seconds_to_srt(caption['end'])}\n"
        )

        f.write(
            f"{caption['text']}\n\n"
        )

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("📝 SHORT-FORM CAPTIONS")
print("=" * 60)

for caption in caption_events:

    print(
        f"{caption['index']:02d}. "
        f"{caption['start']:6.2f}s → "
        f"{caption['end']:6.2f}s  "
        f"{caption['text']}"
    )

print("\n" + "=" * 60)
print("✅ CAPTIONS CREATED")
print("=" * 60)

print(
    f"\n📊 {len(caption_events)} caption events"
)

print(
    f"\n💾 JSON:\n{CAPTIONS_JSON_PATH}"
)

print(
    f"\n💾 SRT:\n{SRT_PATH}"
)

print(
    "\n➡️ Next: Caption styling / video effects"
)

# **CAPTION STYLE SELECTOR**

---



In [ ]:
# @title
# ============================================================
# CELL 14 — CAPTION STYLE SELECTOR
# ============================================================

import os
import json

print("=" * 60)
print("🎨 CAPTION STYLE")
print("=" * 60)

print("""
Choose your caption style:

[1] 🎬 Classic Shorts
    Large white text + black outline

[2] 💥 Bold Impact
    Extra large text + thick outline

[3] 🧊 Clean Minimal
    Clean white text + subtle outline

[4] 🟨 Highlight
    White text with highlighted key words

[5] 📱 TikTok Style
    Large centered captions

[6] 🚫 No Captions
    Skip subtitles completely
""")

CAPTION_STYLE = input(
    "Choose (1-6): "
).strip()

if CAPTION_STYLE not in [
    "1", "2", "3", "4", "5", "6"
]:
    raise ValueError(
        "❌ Invalid option. Choose 1-6."
    )

# ------------------------------------------------------------
# Load captions
# ------------------------------------------------------------

if not os.path.exists(
    CAPTIONS_JSON_PATH
):

    raise FileNotFoundError(
        "❌ captions.json not found.\n"
        "Run Cell 13 first."
    )

with open(
    CAPTIONS_JSON_PATH,
    "r",
    encoding="utf-8"
) as f:

    caption_data = json.load(f)

captions = caption_data["captions"]

# ------------------------------------------------------------
# Style configuration
# ------------------------------------------------------------

styles = {

    "1": {
        "name": "Classic Shorts",
        "font": "Arial",
        "size": 58,
        "bold": True,
        "outline": 5,
        "shadow": 2,
        "alignment": 2,
        "margin_v": 150
    },

    "2": {
        "name": "Bold Impact",
        "font": "Arial",
        "size": 68,
        "bold": True,
        "outline": 7,
        "shadow": 3,
        "alignment": 2,
        "margin_v": 160
    },

    "3": {
        "name": "Clean Minimal",
        "font": "Arial",
        "size": 52,
        "bold": False,
        "outline": 2,
        "shadow": 1,
        "alignment": 2,
        "margin_v": 140
    },

    "4": {
        "name": "Highlight",
        "font": "Arial",
        "size": 60,
        "bold": True,
        "outline": 5,
        "shadow": 2,
        "alignment": 2,
        "margin_v": 150
    },

    "5": {
        "name": "TikTok Style",
        "font": "Arial",
        "size": 62,
        "bold": True,
        "outline": 5,
        "shadow": 2,
        "alignment": 5,
        "margin_v": 0
    }
}

# ------------------------------------------------------------
# No captions
# ------------------------------------------------------------

ASS_PATH = os.path.join(
    OUTPUT_DIR,
    "captions.ass"
)

if CAPTION_STYLE == "6":

    with open(
        ASS_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        f.write("")

    print(
        "\n🚫 Captions disabled."
    )

else:

    style = styles[
        CAPTION_STYLE
    ]

    # --------------------------------------------------------
    # ASS header
    # --------------------------------------------------------

    ass_content = f"""[Script Info]
Title: YTShortsMaker V2
ScriptType: v4.00+
PlayResX: 1080
PlayResY: 1920
ScaledBorderAndShadow: yes

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding

Style: Shorts,{style['font']},{style['size']},&H00FFFFFF,&H00FFFFFF,&H00000000,&H00000000,{-1 if style['bold'] else 0},0,0,0,100,100,0,0,1,{style['outline']},{style['shadow']},{style['alignment']},60,60,{style['margin_v']},1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text

"""

    # --------------------------------------------------------
    # ASS timestamp helper
    # --------------------------------------------------------

    def ass_time(seconds):

        total_cs = int(
            round(seconds * 100)
        )

        hours = total_cs // 360000

        minutes = (
            total_cs % 360000
        ) // 6000

        seconds_part = (
            total_cs % 6000
        ) // 100

        centiseconds = (
            total_cs % 100
        )

        return (
            f"{hours}:"
            f"{minutes:02d}:"
            f"{seconds_part:02d}."
            f"{centiseconds:02d}"
        )

    # --------------------------------------------------------
    # Generate events
    # --------------------------------------------------------

    for caption in captions:

        start = ass_time(
            caption["start"]
        )

        end = ass_time(
            caption["end"]
        )

        text = (
            caption["text"]
            .replace(
                "{",
                "\\{"
            )
            .replace(
                "}",
                "\\}"
            )
        )

        # Highlight style:
        # emphasize first word
        if CAPTION_STYLE == "4":

            words = text.split()

            if len(words) > 0:

                words[0] = (
                    r"{\c&H00FFFF&}"
                    + words[0]
                    + r"{\c&HFFFFFF&}"
                )

                text = " ".join(
                    words
                )

        ass_content += (
            f"Dialogue: 0,"
            f"{start},"
            f"{end},"
            f"Shorts,,"
            f"0,0,0,,"
            f"{text}\n"
        )

    # --------------------------------------------------------
    # Save ASS
    # --------------------------------------------------------

    with open(
        ASS_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            ass_content
        )

# ------------------------------------------------------------
# Save settings
# ------------------------------------------------------------

CAPTION_STYLE_PATH = os.path.join(
    OUTPUT_DIR,
    "caption_style.json"
)

style_data = {

    "style_id":
        CAPTION_STYLE,

    "style_name":
        "No Captions"
        if CAPTION_STYLE == "6"
        else styles[CAPTION_STYLE]["name"],

    "ass_file":
        ASS_PATH
}

with open(
    CAPTION_STYLE_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        style_data,
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 60)
print("✅ CAPTION STYLE CREATED")
print("=" * 60)

print(
    f"\n🎨 Style: "
    f"{style_data['style_name']}"
)

print(
    f"📁 ASS: {ASS_PATH}"
)

print(
    f"💾 Settings: {CAPTION_STYLE_PATH}"
)

print(
    "\n➡️ Next: Video motion/effect selector"
)

# **VIDEO EFFECT SELECTOR**

---



In [ ]:
# @title
# ============================================================
# CELL 15 — VIDEO EFFECT SELECTOR
# ============================================================

import os
import json

print("=" * 60)
print("🎬 VIDEO EFFECT SELECTOR")
print("=" * 60)

print("""
Choose how the images should animate:

[1] 🚫 No Effect
    Static images — no movement

[2] 🔍 Smooth Zoom In
    Slow cinematic zoom toward the image

[3] 🔎 Smooth Zoom Out
    Slow zoom away from the image

[4] ↔️ Gentle Pan
    Smooth horizontal movement

[5] ↕️ Vertical Pan
    Smooth vertical movement

[6] 🎥 Cinematic Zoom + Pan
    Slow zoom combined with subtle movement

[7] 🌫️ Fade Transition
    Smooth fade between scenes

[8] ✨ Zoom + Fade
    Cinematic zoom with fade transitions

[9] 🎬 Ken Burns
    Slow cinematic pan + zoom
""")

EFFECT_CHOICE = input(
    "Choose (1-9): "
).strip()

if EFFECT_CHOICE not in [
    "1", "2", "3", "4", "5",
    "6", "7", "8", "9"
]:
    raise ValueError(
        "❌ Invalid option. Choose 1-9."
    )

# ------------------------------------------------------------
# Effect configuration
# ------------------------------------------------------------

effects = {

    "1": {
        "name": "No Effect",
        "type": "static"
    },

    "2": {
        "name": "Smooth Zoom In",
        "type": "zoom_in"
    },

    "3": {
        "name": "Smooth Zoom Out",
        "type": "zoom_out"
    },

    "4": {
        "name": "Gentle Pan",
        "type": "pan_horizontal"
    },

    "5": {
        "name": "Vertical Pan",
        "type": "pan_vertical"
    },

    "6": {
        "name": "Cinematic Zoom + Pan",
        "type": "zoom_pan"
    },

    "7": {
        "name": "Fade Transition",
        "type": "fade"
    },

    "8": {
        "name": "Zoom + Fade",
        "type": "zoom_fade"
    },

    "9": {
        "name": "Ken Burns",
        "type": "ken_burns"
    }
}

selected_effect = effects[
    EFFECT_CHOICE
]

# ------------------------------------------------------------
# Save effect configuration
# ------------------------------------------------------------

EFFECT_PATH = os.path.join(
    OUTPUT_DIR,
    "video_effect.json"
)

effect_data = {
    "choice": EFFECT_CHOICE,
    "name": selected_effect["name"],
    "type": selected_effect["type"]
}

with open(
    EFFECT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        effect_data,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("✅ VIDEO EFFECT SELECTED")
print("=" * 60)

print(
    f"\n🎬 Effect: "
    f"{selected_effect['name']}"
)

print(
    f"⚙️ Type: "
    f"{selected_effect['type']}"
)

print(
    f"💾 Saved to:\n{EFFECT_PATH}"
)

print(
    "\n➡️ Next: Effect rendering"
)

# **RENDER ANIMATED SCENES**

---



In [ ]:
# @title
# ============================================================
# CELL 16 — RENDER ANIMATED SCENES
# ============================================================

import os
import json
import subprocess
import math

print("=" * 60)
print("🎬 RENDERING ANIMATED SCENES")
print("=" * 60)

# ------------------------------------------------------------
# Load scene data
# ------------------------------------------------------------

if not os.path.exists(SCENES_PATH):
    raise FileNotFoundError(
        "❌ scenes.json not found. Run Cell 12 first."
    )

with open(
    SCENES_PATH,
    "r",
    encoding="utf-8"
) as f:
    scenes_data = json.load(f)

scenes = scenes_data["scenes"]

# ------------------------------------------------------------
# Load selected effect
# ------------------------------------------------------------

if not os.path.exists(EFFECT_PATH):
    raise FileNotFoundError(
        "❌ video_effect.json not found. Run Cell 15 first."
    )

with open(
    EFFECT_PATH,
    "r",
    encoding="utf-8"
) as f:
    effect_data = json.load(f)

EFFECT_TYPE = effect_data["type"]

print(
    f"\n🎬 Selected effect: "
    f"{effect_data['name']}"
)

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

SCENE_VIDEO_DIR = os.path.join(
    OUTPUT_DIR,
    "scenes"
)

os.makedirs(
    SCENE_VIDEO_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# Video settings
# ------------------------------------------------------------

WIDTH = 1080
HEIGHT = 1920
FPS = 30

# Zoom strength
ZOOM_START = 1.00
ZOOM_END = 1.10

# Fade duration
FADE_DURATION = 0.45

# ------------------------------------------------------------
# Render each scene
# ------------------------------------------------------------

rendered_scenes = []

for scene in scenes:

    scene_number = scene["scene_number"]
    image_path = scene["path"]
    duration = float(scene["duration"])

    output_path = os.path.join(
        SCENE_VIDEO_DIR,
        f"scene_{scene_number:02d}.mp4"
    )

    print(
        f"\n🎞️ Scene {scene_number}"
    )

    print(
        f"🖼️ {os.path.basename(image_path)}"
    )

    print(
        f"⏱️ {duration:.2f}s"
    )

    # --------------------------------------------------------
    # Frame count
    # --------------------------------------------------------

    total_frames = max(
        1,
        math.ceil(duration * FPS)
    )

    # --------------------------------------------------------
    # Base input
    # --------------------------------------------------------

    input_args = [
        "ffmpeg",
        "-y",
        "-loop",
        "1",
        "-i",
        image_path,
    ]

    # --------------------------------------------------------
    # Build effect
    # --------------------------------------------------------

    if EFFECT_TYPE == "static":

        vf = (
            f"scale={WIDTH}:{HEIGHT}:"
            f"force_original_aspect_ratio=decrease,"
            f"pad={WIDTH}:{HEIGHT}:(ow-iw)/2:(oh-ih)/2,"
            f"setsar=1,"
            f"fps={FPS}"
        )

    elif EFFECT_TYPE == "zoom_in":

        vf = (
            f"scale={WIDTH*2}:{HEIGHT*2},"
            f"zoompan="
            f"z='min(zoom+0.0008,1.10)':"
            f"x='iw/2-(iw/zoom/2)':"
            f"y='ih/2-(ih/zoom/2)':"
            f"d={total_frames}:"
            f"s={WIDTH}x{HEIGHT}:"
            f"fps={FPS}"
        )

    elif EFFECT_TYPE == "zoom_out":

        vf = (
            f"scale={WIDTH*2}:{HEIGHT*2},"
            f"zoompan="
            f"z='max(1.10-0.0008*on,1.0)':"
            f"x='iw/2-(iw/zoom/2)':"
            f"y='ih/2-(ih/zoom/2)':"
            f"d={total_frames}:"
            f"s={WIDTH}x{HEIGHT}:"
            f"fps={FPS}"
        )

    elif EFFECT_TYPE == "pan_horizontal":

        vf = (
            f"scale={WIDTH*2}:{HEIGHT*2},"
            f"zoompan="
            f"z='1.08':"
            f"x='(iw-iw/zoom)*on/{total_frames}':"
            f"y='ih/2-(ih/zoom/2)':"
            f"d={total_frames}:"
            f"s={WIDTH}x{HEIGHT}:"
            f"fps={FPS}"
        )

    elif EFFECT_TYPE == "pan_vertical":

        vf = (
            f"scale={WIDTH*2}:{HEIGHT*2},"
            f"zoompan="
            f"z='1.08':"
            f"x='iw/2-(iw/zoom/2)':"
            f"y='(ih-ih/zoom)*on/{total_frames}':"
            f"d={total_frames}:"
            f"s={WIDTH}x{HEIGHT}:"
            f"fps={FPS}"
        )

    elif EFFECT_TYPE == "zoom_pan":

        vf = (
            f"scale={WIDTH*2}:{HEIGHT*2},"
            f"zoompan="
            f"z='min(1.0+0.0006*on,1.08)':"
            f"x='(iw-iw/zoom)*on/{total_frames}':"
            f"y='(ih-ih/zoom)*on/{total_frames}':"
            f"d={total_frames}:"
            f"s={WIDTH}x{HEIGHT}:"
            f"fps={FPS}"
        )

    elif EFFECT_TYPE == "fade":

        vf = (
            f"scale={WIDTH}:{HEIGHT}:"
            f"force_original_aspect_ratio=decrease,"
            f"pad={WIDTH}:{HEIGHT}:(ow-iw)/2:(oh-ih)/2,"
            f"setsar=1,"
            f"fade=t=in:st=0:d={FADE_DURATION},"
            f"fade=t=out:"
            f"st={max(0,duration-FADE_DURATION)}:"
            f"d={FADE_DURATION},"
            f"fps={FPS}"
        )

    elif EFFECT_TYPE == "zoom_fade":

        vf = (
            f"scale={WIDTH*2}:{HEIGHT*2},"
            f"zoompan="
            f"z='min(1.0+0.0006*on,1.08)':"
            f"x='iw/2-(iw/zoom/2)':"
            f"y='ih/2-(ih/zoom/2)':"
            f"d={total_frames}:"
            f"s={WIDTH}x{HEIGHT}:"
            f"fps={FPS},"
            f"fade=t=in:st=0:d={FADE_DURATION},"
            f"fade=t=out:"
            f"st={max(0,duration-FADE_DURATION)}:"
            f"d={FADE_DURATION}"
        )

    elif EFFECT_TYPE == "ken_burns":

        vf = (
            f"scale={WIDTH*2}:{HEIGHT*2},"
            f"zoompan="
            f"z='min(1.0+0.0007*on,1.10)':"
            f"x='iw/2-(iw/zoom/2)':"
            f"y='ih/2-(ih/zoom/2)':"
            f"d={total_frames}:"
            f"s={WIDTH}x{HEIGHT}:"
            f"fps={FPS}"
        )

    else:

        raise ValueError(
            f"❌ Unknown effect: {EFFECT_TYPE}"
        )

    # --------------------------------------------------------
    # FFmpeg command
    # --------------------------------------------------------

    command = input_args + [
        "-vf",
        vf,
        "-t",
        str(duration),
        "-r",
        str(FPS),
        "-an",
        "-c:v",
        "libx264",
        "-preset",
        "medium",
        "-crf",
        "18",
        "-pix_fmt",
        "yuv420p",
        output_path
    ]

    # --------------------------------------------------------
    # Run FFmpeg
    # --------------------------------------------------------

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:

        print(
            "❌ FFmpeg error:"
        )

        print(
            result.stderr[-3000:]
        )

        raise RuntimeError(
            f"Scene {scene_number} failed."
        )

    print(
        f"✅ Scene {scene_number}: "
        f"{duration:.2f}s → "
        f"{os.path.basename(output_path)}"
    )

    rendered_scenes.append({
        "scene_number": scene_number,
        "path": output_path,
        "duration": duration
    })

# ------------------------------------------------------------
# Save render information
# ------------------------------------------------------------

RENDERED_SCENES_PATH = os.path.join(
    OUTPUT_DIR,
    "rendered_scenes.json"
)

with open(
    RENDERED_SCENES_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "effect": effect_data,
            "fps": FPS,
            "width": WIDTH,
            "height": HEIGHT,
            "scenes": rendered_scenes
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("\n" + "=" * 60)
print("✅ ALL SCENES RENDERED")
print("=" * 60)

print(
    f"\n🎬 Effect: {effect_data['name']}"
)

print(
    f"🖼️ Scenes: {len(rendered_scenes)}"
)

print(
    f"📐 Resolution: {WIDTH} × {HEIGHT}"
)

print(
    f"🎞️ FPS: {FPS}"
)

print(
    f"\n📁 Saved to:\n{SCENE_VIDEO_DIR}"
)

print(
    f"\n💾 Render data:\n"
    f"{RENDERED_SCENES_PATH}"
)

print(
    "\n➡️ Next: Combine scenes + voiceover"
)

# **COMBINE SCENES + VOICEOVER**

---



In [ ]:
# @title
# ============================================================
# CELL 17 — COMBINE SCENES + VOICEOVER
# ============================================================

import os
import json
import subprocess

print("=" * 60)
print("🎬 COMBINING SCENES + VOICEOVER")
print("=" * 60)

# ------------------------------------------------------------
# Load rendered scenes
# ------------------------------------------------------------

if not os.path.exists(RENDERED_SCENES_PATH):

    raise FileNotFoundError(
        "❌ rendered_scenes.json not found.\n"
        "Run Cell 16 first."
    )

with open(
    RENDERED_SCENES_PATH,
    "r",
    encoding="utf-8"
) as f:

    render_data = json.load(f)

rendered_scenes = render_data["scenes"]

# ------------------------------------------------------------
# Check voiceover
# ------------------------------------------------------------

if not os.path.exists(VOICE_PATH):

    raise FileNotFoundError(
        f"❌ Voiceover not found:\n{VOICE_PATH}\n"
        "Generate the voiceover first."
    )

# ------------------------------------------------------------
# Create concat file
# ------------------------------------------------------------

CONCAT_PATH = os.path.join(
    OUTPUT_DIR,
    "scenes_concat.txt"
)

with open(
    CONCAT_PATH,
    "w",
    encoding="utf-8"
) as f:

    for scene in rendered_scenes:

        path = scene["path"]

        # FFmpeg concat requires escaped paths
        path = path.replace(
            "'",
            "'\\''"
        )

        f.write(
            f"file '{path}'\n"
        )

print(
    f"\n🖼️ Scenes to combine: "
    f"{len(rendered_scenes)}"
)

# ------------------------------------------------------------
# Temporary combined video
# ------------------------------------------------------------

TEMP_VIDEO = os.path.join(
    OUTPUT_DIR,
    "combined_silent.mp4"
)

# ------------------------------------------------------------
# Join scenes
# ------------------------------------------------------------

print("\n🎞️ Joining scenes...")

concat_command = [

    "ffmpeg",
    "-y",

    "-f",
    "concat",

    "-safe",
    "0",

    "-i",
    CONCAT_PATH,

    "-an",

    "-c:v",
    "libx264",

    "-preset",
    "medium",

    "-crf",
    "18",

    "-pix_fmt",
    "yuv420p",

    TEMP_VIDEO
]

result = subprocess.run(
    concat_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

if result.returncode != 0:

    print(
        result.stderr[-4000:]
    )

    raise RuntimeError(
        "❌ Failed to combine scenes."
    )

print(
    "✅ Scenes combined."
)

# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

FINAL_VIDEO_PATH = os.path.join(
    OUTPUT_DIR,
    "short_without_captions.mp4"
)

# ------------------------------------------------------------
# Add voiceover
# ------------------------------------------------------------

print(
    "\n🎙️ Adding voiceover..."
)

audio_command = [

    "ffmpeg",
    "-y",

    "-i",
    TEMP_VIDEO,

    "-i",
    VOICE_PATH,

    "-map",
    "0:v:0",

    "-map",
    "1:a:0",

    "-c:v",
    "copy",

    "-c:a",
    "aac",

    "-b:a",
    "192k",

    # Use the audio as the master duration.
    # Prevents the final spoken words from being clipped.
    "-t",
    "53.18",

    "-movflags",
    "+faststart",

    FINAL_VIDEO_PATH
]

# ------------------------------------------------------------
# IMPORTANT:
# Get actual duration dynamically
# ------------------------------------------------------------

def get_duration(path):

    command = [

        "ffprobe",
        "-v",
        "error",

        "-show_entries",
        "format=duration",

        "-of",
        "default=noprint_wrappers=1:nokey=1",

        path
    ]

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    return float(
        result.stdout.strip()
    )


audio_duration = get_duration(
    VOICE_PATH
)

# Replace hardcoded duration
audio_command[
    audio_command.index("-t") + 1
] = str(audio_duration)

print(
    f"⏱️ Voice duration: "
    f"{audio_duration:.2f}s"
)

result = subprocess.run(
    audio_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

if result.returncode != 0:

    print(
        result.stderr[-4000:]
    )

    raise RuntimeError(
        "❌ Failed to add voiceover."
    )

# ------------------------------------------------------------
# Clean temporary files
# ------------------------------------------------------------

if os.path.exists(
    TEMP_VIDEO
):

    os.remove(
        TEMP_VIDEO
    )

# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

final_duration = get_duration(
    FINAL_VIDEO_PATH
)

print("\n" + "=" * 60)
print("✅ VIDEO + VOICEOVER COMPLETE")
print("=" * 60)

print(
    f"\n🎙️ Audio: "
    f"{audio_duration:.2f}s"
)

print(
    f"🎬 Final video: "
    f"{final_duration:.2f}s"
)

print(
    f"\n📁 Saved to:\n"
    f"{FINAL_VIDEO_PATH}"
)

print(
    "\n➡️ Next: Caption rendering"
)

# **CAPTION RENDERING + FINAL OUTPUT**

---


In [ ]:
# @title
# ============================================================
# CELL 18 — CAPTION RENDERING
# ============================================================

import os
import json
import subprocess
from IPython.display import Video, display

print("=" * 60)
print("📝 RENDERING CAPTIONS")
print("=" * 60)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

if not os.path.exists(FINAL_VIDEO_PATH):

    raise FileNotFoundError(
        "❌ short_without_captions.mp4 not found.\n"
        "Run Cell 17 first."
    )

if not os.path.exists(CAPTION_STYLE_PATH):

    raise FileNotFoundError(
        "❌ caption_style.json not found.\n"
        "Run Cell 14 first."
    )

# ------------------------------------------------------------
# Load caption settings
# ------------------------------------------------------------

with open(
    CAPTION_STYLE_PATH,
    "r",
    encoding="utf-8"
) as f:

    caption_settings = json.load(f)

caption_style = caption_settings["style_name"]

print(
    f"\n🎨 Caption style: {caption_style}"
)

# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

FINAL_OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "final_short.mp4"
)

# ------------------------------------------------------------
# No captions
# ------------------------------------------------------------

if caption_settings["style_id"] == "6":

    print(
        "\n🚫 Captions disabled."
    )

    # Simply copy the video
    command = [
        "ffmpeg",
        "-y",

        "-i",
        FINAL_VIDEO_PATH,

        "-c",
        "copy",

        "-movflags",
        "+faststart",

        FINAL_OUTPUT_PATH
    ]

# ------------------------------------------------------------
# Render ASS captions
# ------------------------------------------------------------

else:

    if not os.path.exists(ASS_PATH):

        raise FileNotFoundError(
            "❌ captions.ass not found.\n"
            "Run Cell 14 again."
        )

    print(
        "\n🔥 Burning captions into video..."
    )

    # FFmpeg subtitles filter
    #
    # ASS already contains the 1080x1920
    # positioning and styling information.

    subtitle_filter = (
        f"subtitles="
        f"'{ASS_PATH.replace(':', '\\:')}'"
    )

    command = [

        "ffmpeg",
        "-y",

        "-i",
        FINAL_VIDEO_PATH,

        "-vf",
        subtitle_filter,

        "-c:v",
        "libx264",

        "-preset",
        "medium",

        "-crf",
        "18",

        "-pix_fmt",
        "yuv420p",

        "-c:a",
        "copy",

        "-movflags",
        "+faststart",

        FINAL_OUTPUT_PATH
    ]

# ------------------------------------------------------------
# Run FFmpeg
# ------------------------------------------------------------

result = subprocess.run(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

if result.returncode != 0:

    print(
        "\n❌ FFmpeg error:\n"
    )

    print(
        result.stderr[-5000:]
    )

    raise RuntimeError(
        "Caption rendering failed."
    )

# ------------------------------------------------------------
# Verify output
# ------------------------------------------------------------

if not os.path.exists(
    FINAL_OUTPUT_PATH
):

    raise RuntimeError(
        "❌ Final video was not created."
    )

# ------------------------------------------------------------
# Get final duration
# ------------------------------------------------------------

def get_video_duration(path):

    command = [

        "ffprobe",
        "-v",
        "error",

        "-show_entries",
        "format=duration",

        "-of",
        "default=noprint_wrappers=1:nokey=1",

        path
    ]

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    return float(
        result.stdout.strip()
    )


final_duration = get_video_duration(
    FINAL_OUTPUT_PATH
)

# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("🎬 FINAL SHORT CREATED")
print("=" * 60)

print(
    f"\n⏱️ Duration: "
    f"{final_duration:.2f}s"
)

print(
    f"🎨 Captions: "
    f"{caption_style}"
)

print(
    "\n📐 Format: 1080 × 1920"
)

print(
    "\n📁 Final video:"
)

print(
    FINAL_OUTPUT_PATH
)

print("\n" + "=" * 60)
print("✅ READY FOR YOUTUBE SHORTS")
print("=" * 60)

# ------------------------------------------------------------
# Preview
# ------------------------------------------------------------

display(
    Video(
        FINAL_OUTPUT_PATH,
        embed=True,
        width=360
    )
)